# Apex Retail Intelligence

## Notebook 02 - Bronze Layer

### Objective

This notebook implements the Bronze Layer of the Medallion Architecture.

### Steps

- Read Landing Parquet files
- Preserve all source data
- Add ingestion metadata
- Write Delta tables
- Historical and Incremental data stored separately

### Notes

- No data cleansing
- No deduplication
- No datatype conversion
- Bronze is append-only for incremental data

In [0]:
# ============================================================
# Imports
# ============================================================

from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp

In [0]:
# ============================================================
# Create Spark Session
# ============================================================

spark = (
    SparkSession.builder
    .appName("Apex Retail Intelligence - Bronze Layer")
    .getOrCreate()
)

In [0]:
# ============================================================
# Project Configuration
# ============================================================

BASE_PATH = "/Volumes/workspace/default/apex_retail_data"

LANDING_PATH = f"{BASE_PATH}/landing"
BRONZE_PATH = f"{BASE_PATH}/bronze"

In [0]:
def read_landing_data(path: str):
    """
    Reads Landing Parquet data.
    """
    return spark.read.parquet(path)

In [0]:
def add_ingestion_metadata(df):
    """
    Adds ingestion timestamp.

    This timestamp records when the data entered
    the Bronze layer.
    """

    return df.withColumn(
        "ingested_at",
        current_timestamp()
    )

In [0]:
def write_bronze_data(
    df,
    output_path: str,
    write_mode: str
):
    """
    Writes data to the Bronze layer in Delta format.

    Parameters
    ----------
    overwrite
        Used for Historical Load

    append
        Used for Incremental Load
    """

    (
        df.write
        .format("delta")
        .mode(write_mode)
        .save(output_path)
    )

    print(f"Bronze table written successfully -> {output_path}")

In [0]:
def process_bronze_dataset(
    dataset_name: str,
    landing_path: str,
    bronze_path: str,
    write_mode: str
):
    """
    End-to-End Bronze Processing

    Steps
    -----
    1. Read Landing Parquet
    2. Add ingestion timestamp
    3. Write Delta table
    """

    print("=" * 70)
    print(f"Processing : {dataset_name}")
    print("=" * 70)

    # Read Landing Data
    df = read_landing_data(landing_path)

    # Add Metadata
    bronze_df = add_ingestion_metadata(df)

    # Write Bronze Delta
    write_bronze_data(
        bronze_df,
        bronze_path,
        write_mode
    )

    print(f"{dataset_name} completed successfully.\n")

In [0]:
process_bronze_dataset(
    dataset_name="Customer Historical",
    landing_path=f"{LANDING_PATH}/customer/historical",
    bronze_path=f"{BRONZE_PATH}/customer/historical",
    write_mode="overwrite"
)

Processing : Customer Historical
Bronze table written successfully -> /Volumes/workspace/default/apex_retail_data/bronze/customer/historical
Customer Historical completed successfully.



In [0]:
process_bronze_dataset(
    dataset_name="Customer Incremental",
    landing_path=f"{LANDING_PATH}/customer/incremental",
    bronze_path=f"{BRONZE_PATH}/customer/incremental",
    write_mode="append"
)

Processing : Customer Incremental
Bronze table written successfully -> /Volumes/workspace/default/apex_retail_data/bronze/customer/incremental
Customer Incremental completed successfully.



In [0]:
spark.range(5).write.format("delta").mode("overwrite").save(
    f"{BASE_PATH}/bronze/delta_test"
)

In [0]:
process_bronze_dataset(
    dataset_name="Customer Historical",
    landing_path=f"{LANDING_PATH}/customer/historical",
    bronze_path=f"{BRONZE_PATH}/customer/historical",
    write_mode="overwrite"
)

process_bronze_dataset(
    dataset_name="Product Historical",
    landing_path=f"{LANDING_PATH}/product/historical",
    bronze_path=f"{BRONZE_PATH}/product/historical",
    write_mode="overwrite"
)

process_bronze_dataset(
    dataset_name="Sales Historical",
    landing_path=f"{LANDING_PATH}/sales/historical",
    bronze_path=f"{BRONZE_PATH}/sales/historical",
    write_mode="overwrite"
)

Processing : Customer Historical
Bronze table written successfully -> /Volumes/workspace/default/apex_retail_data/bronze/customer/historical
Customer Historical completed successfully.

Processing : Product Historical
Bronze table written successfully -> /Volumes/workspace/default/apex_retail_data/bronze/product/historical
Product Historical completed successfully.

Processing : Sales Historical
Bronze table written successfully -> /Volumes/workspace/default/apex_retail_data/bronze/sales/historical
Sales Historical completed successfully.



In [0]:
process_bronze_dataset(
    dataset_name="Customer Incremental",
    landing_path=f"{LANDING_PATH}/customer/incremental",
    bronze_path=f"{BRONZE_PATH}/customer/incremental",
    write_mode="overwrite"
)

process_bronze_dataset(
    dataset_name="Product Incremental",
    landing_path=f"{LANDING_PATH}/product/incremental",
    bronze_path=f"{BRONZE_PATH}/product/incremental",
    write_mode="append"
)

process_bronze_dataset(
    dataset_name="Sales Incremental",
    landing_path=f"{LANDING_PATH}/sales/incremental",
    bronze_path=f"{BRONZE_PATH}/sales/incremental",
    write_mode="append"
)

Processing : Customer Incremental
Bronze table written successfully -> /Volumes/workspace/default/apex_retail_data/bronze/customer/incremental
Customer Incremental completed successfully.

Processing : Product Incremental
Bronze table written successfully -> /Volumes/workspace/default/apex_retail_data/bronze/product/incremental
Product Incremental completed successfully.

Processing : Sales Incremental
Bronze table written successfully -> /Volumes/workspace/default/apex_retail_data/bronze/sales/incremental
Sales Incremental completed successfully.



In [0]:
# ============================================================
# Verify Bronze Layer
# ============================================================

bronze_datasets = [
    ("Customer Historical", f"{BRONZE_PATH}/customer/historical"),
    ("Product Historical", f"{BRONZE_PATH}/product/historical"),
    ("Sales Historical", f"{BRONZE_PATH}/sales/historical"),
    ("Customer Incremental", f"{BRONZE_PATH}/customer/incremental"),
    ("Product Incremental", f"{BRONZE_PATH}/product/incremental"),
    ("Sales Incremental", f"{BRONZE_PATH}/sales/incremental")
]

for dataset_name, path in bronze_datasets:

    df = (
        spark.read
        .format("delta")
        .load(path)
    )

    print("=" * 70)
    print(f"{dataset_name}")
    print(f"Rows    : {df.count()}")
    print(f"Columns : {len(df.columns)}")

    # Verify metadata column exists
    assert "ingested_at" in df.columns, \
        f"'ingested_at' column missing in {dataset_name}"

print("\n✅ Bronze Layer verification completed successfully.")

Customer Historical
Rows    : 1052
Columns : 15
Product Historical
Rows    : 1043
Columns : 17
Sales Historical
Rows    : 1002
Columns : 20
Customer Incremental
Rows    : 1053
Columns : 20
Product Incremental
Rows    : 2082
Columns : 18
Sales Incremental
Rows    : 2000
Columns : 20

✅ Bronze Layer verification completed successfully.
